# 02 — Feature Engineering

Notebook 01 gave us a raw candidate pool (`data/processed/candidates_c81_kdlh.csv`):
230 named/unnamed landmarks along the C81 → KDLH corridor, tagged with their
OSM category and their position relative to the direct route. That's still
just "things OSM knows about near the line" — this notebook turns each
candidate into a numeric feature vector describing how *good a visual
waypoint* it plausibly is, which is what Notebook 03's model will actually
learn from.

Features built here:
1. `cross_track_nm` / `along_track_nm` — carried over from Notebook 01 (route geometry)
2. `log_size` — log-scaled footprint area
3. one-hot columns for `category` (feature type)
4. `elevation_prominence_m` — local relief from USGS 3DEP (does it stick up?)
5. `name_uniqueness` — does its name collide with another candidate on the route?
6. `nn_dist_nm` — distance to the nearest other candidate ("clutter")

Output: `data/processed/features_c81_kdlh.parquet`.


In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import json
import numpy as np
import pandas as pd

from vfr import features, elevation

pd.set_option("display.max_columns", None)


## Step 1 — Load Notebook 01's output

`tags` was serialized to JSON so it would survive a CSV round-trip; parse it
back into a dict now that we're in a fresh kernel.


In [2]:
IN_PATH = PROJECT_ROOT / "data" / "processed" / "candidates_c81_kdlh.csv"
df = pd.read_csv(IN_PATH)
df["tags"] = df["tags"].apply(json.loads)
print(df.shape)
df.head()


(426, 11)


,osm_id,osm_type,category,name,lat,lon,bbox_area_m2,tags,cross_track_nm,along_track_nm,within_preferred_corridor
0,151331909,node,town,Superior,46.720774,-92.104080,0.0,"{'capital': '6', 'ele': '196', 'gnis:feature_i...",-0.906271,315.222733,False
1,153420031,node,town,Montello,43.791861,-89.328819,0.0,"{'capital': '6', 'ele': '242', 'gnis:feature_i...",0.062879,103.874318,True
2,153546173,node,town,Round Lake,42.353355,-88.093414,0.0,"{'ele': '243', 'gnis:feature_id': '416994', 'n...",0.174845,1.919655,True
3,153566547,node,town,Round Lake Beach,42.371688,-88.090081,0.0,"{'ele': '233', 'gnis:feature_id': '416995', 'n...",0.877854,2.779455,False
4,353870611,node,lake_or_pond,Big Falls Flowage,45.555885,-90.960424,0.0,"{'ele': '372', 'gnis:feature_id': '1561722', '...",-0.602629,230.635294,False


## Step 2 — Size feature

`bbox_area_m2` is heavily right-skewed (a handful of real lakes vs. hundreds
of small features), which would let a couple of huge lakes dominate a
linear model's size coefficient. `log1p` compresses that range; point
features (towers, towns — no polygon, so `bbox_area_m2 == 0`) map to 0
either way.


In [3]:
df["log_size"] = features.log_size_feature(df["bbox_area_m2"])
df[["category", "bbox_area_m2", "log_size"]].groupby("category").agg(["mean", "max"])


bbox_area_m2                 log_size           
                      mean           max       mean        max
category                                                      
intersection  0.000000e+00  0.000000e+00   0.000000   0.000000
lake_or_pond  6.849433e+06  6.649056e+08  12.061932  20.315156
quarry        1.226276e+06  8.994252e+06  11.671512  16.012096
railroad      0.000000e+00  0.000000e+00   0.000000   0.000000
river         0.000000e+00  0.000000e+00   0.000000   0.000000
stadium       1.250964e+04  2.501929e+04   5.063721  10.127442
tower         0.000000e+00  0.000000e+00   0.000000   0.000000
town          0.000000e+00  0.000000e+00   0.000000   0.000000
water_tower   1.058452e+02  6.975232e+02   1.651855   6.548968
wind_farm     6.760877e+07  6.760877e+07  18.029248  18.029248

## Step 3 — Feature type (one-hot)

`category` is a plain string right now (`lake_or_pond`, `tower`, `town`,
...). Most models in Notebook 03 need numeric input, so one-hot encode it —
`pandas.get_dummies` is the standard tool for this on a DataFrame.


In [4]:
category_dummies = pd.get_dummies(df["category"], prefix="category")
df = pd.concat([df, category_dummies], axis=1)
category_dummies.sum().sort_values(ascending=False)


category_lake_or_pond    166
category_tower            83
category_intersection     75
category_river            46
category_railroad         22
category_water_tower      17
category_quarry           10
category_town              4
category_stadium           2
category_wind_farm         1
dtype: int64

## Step 4 — Elevation prominence (USGS 3DEP)

A water tower on a hilltop or a bluff over a river is easier to spot from
the air than the same feature sitting on dead-flat ground. `vfr.elevation`
samples real point elevation from USGS's public Elevation Point Query
Service — at the candidate's coordinates, and at four points 1 nm out to
the N/E/S/W — and returns `candidate_elevation - mean(ring_elevation)` as a
rough local-prominence score.

The public EPQS endpoint is slow (a fraction of a second per point, times
~1,150 points for 230 candidates), so this is parallelized across threads
and cached to `data/raw/elevation_cache.csv`. **First run takes several
minutes; re-runs are instant** since every point gets cached by
coordinate.


In [5]:
import time

candidate_points = list(zip(df["lat"], df["lon"]))
t0 = time.time()
df["elevation_prominence_m"] = elevation.elevation_prominence_m(candidate_points)
print(f"Elevation lookups done in {time.time() - t0:.0f}s")
df["elevation_prominence_m"].describe()


Elevation lookups done in 144s


count    426.000000
mean       0.273845
std       17.369987
min      -70.135147
25%       -7.516943
50%       -1.951139
75%        2.517473
max       91.092041
Name: elevation_prominence_m, dtype: float64

## Step 5 — Name uniqueness

A checkpoint list is only useful if a pilot can tell which "Long Lake" it
means. `name_uniqueness` is `1 / (count of candidates sharing this name)` —
1.0 for a name that appears once, 0.5 if it appears twice, NaN if the
candidate has no name at all (handled separately, not penalized the same
way as a *colliding* name).


In [6]:
df["name_uniqueness"] = features.name_uniqueness(df["name"])
df[["name", "name_uniqueness"]][df["name"].notna()].drop_duplicates().sort_values("name_uniqueness").head(10)


,name,name_uniqueness
368,WINDMILL (RANDOLPH),0.062500
349,TOWER (DULUTH),0.083333
218,Montello River,0.125000
245,CN Superior Subdivision,0.142857
392,TOWER (COLOMA),0.166667
202,North Fork Popple River,0.200000
403,TOWER (MARSHFIELD),0.250000
211,Little Eau Pleine River,0.333333
223,Namekagon River,0.333333
235,Amnicon River,0.333333


## Step 6 — Nearest-neighbor clutter distance

Two candidates 0.1 nm apart are effectively the same waypoint decision —
whichever a model prefers, having its near-duplicate sitting right next to
it doesn't add information and could bias spacing logic in Notebook 07.
`nn_dist_nm` is the great-circle distance from each candidate to its
closest neighbor in the whole candidate pool (not just same-category).


In [7]:
df["nn_dist_nm"] = features.nearest_neighbor_distance_nm(df["lat"], df["lon"])
df["nn_dist_nm"].describe()


count    426.000000
mean       0.516257
std        0.617870
min        0.000604
25%        0.159308
50%        0.342486
75%        0.631459
max        4.637528
Name: nn_dist_nm, dtype: float64

## Step 7 — Assemble the feature table and save

Keep identifying columns (name, category, coordinates) alongside the
numeric features — Notebook 03 will need to drop the identifiers before
fitting a model, but we want them around for labeling and for reading
results back in a human-readable way.

Parquet (not CSV) from here on: it round-trips dtypes and nested-free
columns cleanly, which matters once we're passing this into scikit-learn
and, in Notebook 06, Spark.


In [8]:
feature_cols = [
    "cross_track_nm",
    "along_track_nm",
    "within_preferred_corridor",
    "log_size",
    "elevation_prominence_m",
    "name_uniqueness",
    "nn_dist_nm",
] + list(category_dummies.columns)

id_cols = ["osm_id", "osm_type", "category", "name", "lat", "lon"]

out_df = df[id_cols + feature_cols].copy()

OUT_PATH = PROJECT_ROOT / "data" / "processed" / "features_c81_kdlh.parquet"
out_df.to_parquet(OUT_PATH, index=False)
print(f"Saved {len(out_df)} rows x {len(feature_cols)} features to {OUT_PATH}")
out_df.head()


Saved 426 rows x 17 features to data/processed/features_c81_kdlh.parquet


,osm_id,osm_type,category,name,lat,lon,cross_track_nm,along_track_nm,within_preferred_corridor,log_size,elevation_prominence_m,name_uniqueness,nn_dist_nm,category_intersection,category_lake_or_pond,category_quarry,category_railroad,category_river,category_stadium,category_tower,category_town,category_water_tower,category_wind_farm
0,151331909,node,town,Superior,46.720774,-92.104080,-0.906271,315.222733,False,0.0,2.880959,1.0,0.008846,False,False,False,False,False,False,False,True,False,False
1,153420031,node,town,Montello,43.791861,-89.328819,0.062879,103.874318,True,0.0,-15.898533,1.0,0.062157,False,False,False,False,False,False,False,True,False,False
2,153546173,node,town,Round Lake,42.353355,-88.093414,0.174845,1.919655,True,0.0,6.072186,1.0,0.035830,False,False,False,False,False,False,False,True,False,False
3,153566547,node,town,Round Lake Beach,42.371688,-88.090081,0.877854,2.779455,False,0.0,-4.861698,1.0,0.334172,False,False,False,False,False,False,False,True,False,False
4,353870611,node,lake_or_pond,Big Falls Flowage,45.555885,-90.960424,-0.602629,230.635294,False,0.0,-9.940987,1.0,0.197171,False,True,False,False,False,False,False,False,False,False


## Step 8 — Sanity check

A quick look at whether the features line up with intuition: prominent
towers/quarries should skew toward higher `elevation_prominence_m` than
lakes (which sit in low ground by definition), and `log_size` should
clearly separate towns/lakes from point features.


In [9]:
out_df.groupby("category")[["log_size", "elevation_prominence_m", "nn_dist_nm"]].mean().sort_values(
    "elevation_prominence_m", ascending=False
)


,log_size,elevation_prominence_m,nn_dist_nm
category,,,
tower,0.000000,18.111588,0.469568
quarry,11.671512,4.824832,0.550333
wind_farm,18.029248,4.716911,0.524561
water_tower,1.651855,1.957616,0.438112
stadium,5.063721,1.230049,0.090864
intersection,0.000000,-1.079801,0.483293
town,0.000000,-2.951772,0.110251
lake_or_pond,12.061932,-4.472209,0.540945
railroad,0.000000,-6.980863,0.537087
